In [2]:
import pandas as pd

## Задание 1
Напишите функцию, которая классифицирует фильмы из материалов занятия по правилам:

- оценка 2 и ниже — низкий рейтинг;
- оценка 4 и ниже — средний рейтинг;
- оценка 4.5 и 5 — высокий рейтинг.

Результат классификации запишите в столбец class.


In [3]:
movies_df = pd.read_csv('ml-latest-small/movies.csv')
ratings_df = pd.read_csv('ml-latest-small/ratings.csv')
movies_ratings_df = ratings_df.merge(movies_df, on='movieId')
movies_ratings_df.head(3)

,userId,movieId,rating,timestamp,title,genres
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller


In [4]:
def get_rating_title(row):
    if row['rating'] <= 2.0:
        return 'низкий рейтинг'
    elif row['rating'] < 4.5:
        return 'средний рейтинг'
    elif row['rating'] >= 4.5:
        return 'высокий рейтинг'
    else: return None
    
movies_ratings_df['class'] = movies_ratings_df.apply(get_rating_title, axis=1)
movies_ratings_df.head(5)

,userId,movieId,rating,timestamp,title,genres,class
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama,средний рейтинг
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical,средний рейтинг
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller,средний рейтинг
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,низкий рейтинг
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama,средний рейтинг


## Задание 2

Используйте файл keywords.csv.

Нужно написать гео-классификатор, который каждой строке сможет выставить географическую принадлежность определённому региону. Т. е. если поисковый запрос содержит название города региона, то в столбце ‘region’ пишется название этого региона. Если поисковый запрос не содержит названия города, то ставим ‘undefined’.

Правила распределения по регионам Центр, Северо-Запад и Дальний Восток:

geo_data = {
    'Центр': ['москва', 'тула', 'ярославль'],
    'Северо-Запад': ['петербург', 'псков', 'мурманск'],
    'Дальний Восток': ['владивосток', 'сахалин', 'хабаровск']
    }

Результат классификации запишите в отдельный столбец region.

In [5]:
geo_data = {
    'Центр': ['москва', 'тула', 'ярославль'],
    'Северо-Запад': ['петербург', 'псков', 'мурманск'],
    'Дальний Восток': ['владивосток', 'сахалин', 'хабаровск']
    }
keywords_df = pd.read_csv('ml-latest-small/keywords.csv')
keywords_df.head()

,keyword,shows
0,вк,64292779
1,одноклассники,63810309
2,порно,41747114
3,ютуб,39995567
4,вконтакте,21014195


In [5]:
keywords_df[keywords_df['keyword'].str.contains('моск', case=False)].head(5)

,keyword,shows
127,авито москва,979292
143,эхо москвы,889657
155,школьный портал московской области,836987
197,погода в москве,745745
414,погода в москве на 14 дней,400914


In [9]:
def get_region(row, geo_data):
    keyword = row['keyword'].lower()
    for region, cities in geo_data.items():
        for city in cities:
            if city in keyword:
                return region
    else: return 'undefined'

In [13]:
keywords_work_df = keywords_df.copy()

keywords_work_df['region'] = keywords_work_df.apply(lambda row: get_region(row, geo_data), axis=1)
keywords_work_df[keywords_work_df['keyword'].str.contains('сахалин', case=False)].head(5)

,keyword,shows,region
21445,сахалин,14494,Дальний Восток
23547,южно сахалинск,13386,Дальний Восток
41988,сетевой город образование сахалинская область,7817,Дальний Восток
44291,сетевой город южно сахалинск,7515,Дальний Восток
58451,погода в южно сахалинске,6053,Дальний Восток


## Задание 3 (бонусное)

Есть мнение, что раньше снимали настоящее кино, не то что сейчас. Ваша задача — проверить это утверждение, используя файлы с рейтингами фильмов из прошлого домашнего занятия: файл movies.csv и ratings.csv из базы. Нужно проверить, верно ли, что с ростом года выпуска фильма его средний рейтинг становится ниже.

Вы не будете затрагивать субьективные факторы выставления этих рейтингов, а пройдётесь по алгоритму:

1. В переменную years запишите список из всех годов с 1950 по 2010 года.

2. Напишите функцию production_year, которая каждой строке из названия фильма выставляет год выпуска. Не все названия фильмов содержат год выпуска в одинаковом формате, поэтому используйте алгоритм:
    - для каждой строки пройдите по всем годам списка years;
    - если номер года присутствует в названии фильма, то функция возвращает этот год, как год выпуска;
    - если ни один из номеров года списка years не встретился в названии фильма, то возвращается 1900 год.
3. Запишите год выпуска фильма по алгоритму пункта 2 в новый столбец ‘year’.

4. Посчитайте средний рейтинг всех фильмов для каждого значения столбца ‘year’ и отсортируйте результат по убыванию рейтинга.

In [131]:
hm3_movies_ratings_df = movies_ratings_df.copy()
hm3_movies_ratings_df.head()

,userId,movieId,rating,timestamp,title,genres,class
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama,средний рейтинг
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical,средний рейтинг
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller,средний рейтинг
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,низкий рейтинг
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama,средний рейтинг


In [113]:
import re
test = 'Dangerous Minds (1995)'

year_patern = re.findall(r'(\d{4})',test)
year_patern

['1995']

In [135]:
def productin_year(row):
    years = [x for x in range(1950, 2010 + 1)]
    productin_year_patern = re.findall(r'\d{4}',row['title'])
    
    if len(productin_year_patern)>0:
        productin_year_patern = int(productin_year_patern[0]) 
        
        if productin_year_patern in years:
            return productin_year_patern
        else: return 1900
    else: return 1900


In [138]:
hm3_movies_ratings_df['year'] = hm3_movies_ratings_df.apply(productin_year, axis=1)
hm3_movies_ratings_df.head(10)

,userId,movieId,rating,timestamp,title,genres,class,year
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama,средний рейтинг,1995
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical,средний рейтинг,1900
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller,средний рейтинг,1996
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,низкий рейтинг,1981
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama,средний рейтинг,1989
5,1,1263,2.0,1260759151,"Deer Hunter, The (1978)",Drama|War,низкий рейтинг,1978
6,1,1287,2.0,1260759187,Ben-Hur (1959),Action|Adventure|Drama,низкий рейтинг,1959
7,1,1293,2.0,1260759148,Gandhi (1982),Drama,низкий рейтинг,1982
8,1,1339,3.5,1260759125,Dracula (Bram Stoker's Dracula) (1992),Fantasy|Horror|Romance|Thriller,средний рейтинг,1992
9,1,1343,2.0,1260759131,Cape Fear (1991),Thriller,низкий рейтинг,1991


In [142]:
hm3_movies_ratings_df.groupby('year')['rating'].mean().sort_values(ascending=False)

year
1957    4.014241
1972    4.011136
1952    4.000000
1974    3.999058
1954    3.994220
          ...   
2005    3.448434
2003    3.445843
1996    3.422675
1997    3.416934
2000    3.353602
Name: rating, Length: 62, dtype: float64